### **FIRST STEP. Carga y procesamiento de los datos.**

El cuaderno incluye: 

- Carga de los datos así como su estructuración en un conjunto de datos más accesible por los diferentes modelos.
- Enriquecimiento/aumentación de datos y de nuevo estructuración de los datos aumentados en un conjunto de datos más accesible por los diferentes modelos.

In [1]:
# Importación de todas las librerías necesarias en la sección 
from pathlib import Path
import csv
from collections import Counter
import pandas as pd
from typing import List, Tuple
from PIL import Image
import numpy as np
from tf_keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

Se carga el conjunto de datos creando un datastet con el path de cada imagen y el nombre de la carpeta que la contiene como su etiqueta.

In [2]:
# Función para construir un CSV de imágenes (path, etiqueta) desde un directorio raíz
def build_image_csv(root_dir: Path | str,
                    output_csv: Path | str,
                    image_exts: set = {'.jpg', '.jpeg', '.png'},
                    write_csv: bool = True) -> pd.DataFrame:
    """
    Recorre root_dir buscando imágenes y crea un CSV con columnas (filepath,label).
    Devuelve un DataFrame con los resultados.
    """
    root = Path(root_dir)
    output_csv = Path(output_csv)

    if not root.exists():
        raise FileNotFoundError(f'No se encuentra el directorio: {root.resolve()}')

    exts = {e.lower() for e in image_exts}
    rows = []

    for file in root.rglob('*'):
        if file.is_file() and file.suffix.lower() in exts:
            label = file.parent.name
            rows.append([str(file), label])

    if not rows:
        raise RuntimeError('No se encontraron imágenes con las extensiones indicadas en el directorio especificado.')

    df = pd.DataFrame(rows, columns=['filepath', 'label'])

    if write_csv:
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_csv, index=False, encoding='utf-8')
        print(f'Dataset {output_csv} guardado con éxito.\n')

    print(f'Total de imágenes: {len(rows)}\n')

    return df


# Uso en el conjunto original de imágenes:
data = build_image_csv(
    root_dir='.\Images',
    output_csv='synchronized_swimming.csv'
)

# Imprimir algunas estadísticas del DataFrame resultante
print("Número de imágenes por clase:")
print(data['label'].value_counts())
print()
print("Primeras 5 filas del DataFrame:")
data.head()

Dataset synchronized_swimming.csv guardado con éxito.

Total de imágenes: 263

Número de imágenes por clase:
label
Knight                             63
Fishtail                           59
Bent Knee Vertical                 52
Double Leg Vertical                50
Bent Knee Surface Arch Position    39
Name: count, dtype: int64

Primeras 5 filas del DataFrame:


<>:41: SyntaxWarning: invalid escape sequence '\I'
<>:41: SyntaxWarning: invalid escape sequence '\I'
C:\Users\PaulaBallesteros\AppData\Local\Temp\ipykernel_28200\2219998160.py:41: SyntaxWarning: invalid escape sequence '\I'
  root_dir='.\Images',


,filepath,label
0,Images\Bent Knee Surface Arch Position\IMG_138...,Bent Knee Surface Arch Position
1,Images\Bent Knee Surface Arch Position\IMG_138...,Bent Knee Surface Arch Position
2,Images\Bent Knee Surface Arch Position\IMG_139...,Bent Knee Surface Arch Position
3,Images\Bent Knee Surface Arch Position\IMG_139...,Bent Knee Surface Arch Position
4,Images\Bent Knee Surface Arch Position\IMG_139...,Bent Knee Surface Arch Position


Se definen funciones de enriquecimiento/aumentación.

In [ ]:
# Funciones para aumentación de datos

def _ensure_rgb(image: Image.Image) -> Image.Image:
    """Convierte la imagen a RGB si no lo está."""
    return image.convert('RGB') if image.mode != 'RGB' else image


def create_augmentation_generator(seed: int = 42) -> ImageDataGenerator:
    """
    Crea un generador de aumentación de datos que genera variaciones naturales.
    """
    return ImageDataGenerator(
        rescale=1./255.,
        rotation_range=20,              # Rotaciones leves
        width_shift_range=0.1,          # Desplazamientos horizontales
        height_shift_range=0.1,         # Desplazamientos verticales
        zoom_range=[0.85, 1.15],        # Zoom ligero in/out
        horizontal_flip=True,           # Flip horizontal permitido
        vertical_flip=False,            # NO flip vertical para mantener orientación natural
        fill_mode='nearest',
        brightness_range=[0.7, 1.3],    # Variación de iluminación
    )



def random_augmentation(image: Image.Image, datagen: ImageDataGenerator, seed: int = None) -> Image.Image:
    """
    Aplica aumentación de datos usando Keras ImageDataGenerator.
    """
    # Convertir a array numpy
    img_array = np.array(image).astype('float32')
    
    # Expandir dimensiones para batch (1, height, width, channels)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Aplicar transformación
    aug_iter = datagen.flow(img_batch, batch_size=1, seed=seed)
    aug_array = next(aug_iter)[0]
    
    # Convertir de vuelta a PIL (ya está en rango [0,1])
    aug_array = (aug_array * 255).astype('uint8')
    return Image.fromarray(aug_array)


def load_dataset_csv(csv_path: Path) -> List[Tuple[Path, str]]:
    """Carga el dataset desde un archivo CSV."""
    items: List[Tuple[Path, str]] = []
    with csv_path.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            fp = Path(row['filepath'])
            label = row['label']
            if fp.exists():
                items.append((fp, label))
    
    if not items:
        raise RuntimeError('El CSV no contiene rutas válidas o no existen los archivos.')
    
    return items


def augment_dataset(
    csv_path: Path | str = Path('dataset.csv'),
    output_root: Path | str = Path('Augmented'),
    num_aug_per_image: int = 3,
    seed: int | None = 42,
    keep_original_structure: bool = True,
) -> None:
    """
    Genera variaciones aumentadas para cada imagen del dataset.
    
    Args:
        csv_path: Ruta al archivo CSV con las imágenes
        output_root: Directorio de salida para imágenes aumentadas
        num_aug_per_image: Número de aumentaciones por imagen
        seed: Semilla para reproducibilidad
        keep_original_structure: Si mantener la estructura de directorios por etiqueta
    """
    csv_path = Path(csv_path)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    # Crear el generador de aumentación
    datagen = create_augmentation_generator(seed=seed)
    
    # Cargar dataset
    items = load_dataset_csv(csv_path)

    saved = 0
    skipped = 0

    for src_path, label in items:
        try:
            with Image.open(src_path) as im:
                im = _ensure_rgb(im)
                original_size = im.size

                # Directorio de salida
                out_dir = output_root / label if keep_original_structure else output_root
                out_dir.mkdir(parents=True, exist_ok=True)

                stem = src_path.stem
                
                # Generar múltiples aumentaciones
                for k in range(1, num_aug_per_image + 1):
                    current_seed = seed + k if seed is not None else None
                    
                    # Aplicar aumentación
                    aug_img = random_augmentation(im, datagen, seed=current_seed)
                    
                    # Mantener tamaño original
                    if aug_img.size != original_size:
                        aug_img = aug_img.resize(original_size, Image.Resampling.BICUBIC)

                    # Generar nombre único
                    out_name = f"{stem}_aug{k}.jpg"
                    out_file = out_dir / out_name

                    # Evitar sobrescritura
                    attempt = 1
                    while out_file.exists():
                        out_name = f"{stem}_aug{k}_{attempt}.jpg"
                        out_file = out_dir / out_name
                        attempt += 1

                    # Guardar imagen
                    aug_img.save(out_file, format='JPEG', quality=90, optimize=True)
                    saved += 1
                    
        except Exception as e:
            skipped += 1
            print(f"[ERROR] No se pudo procesar {src_path}: {e}")

    print(f"Guardadas {saved} imágenes aumentadas en '{output_root}'.")
    print(f"Omitidas: {skipped}")


# Generar imagenes aumentadas
augment_dataset(
    csv_path='synchronized_swimming.csv',
    output_root='Augmented2',
    num_aug_per_image=25,
    seed=42
)

Guardadas 6575 imágenes aumentadas en 'Augmented'.
Omitidas: 0


Estructuración del nuevo conjunto de datos aumentado.

In [4]:
## Convierto todas las imágenes aumentadas en un dataset igual que con las originales:
data_aug = build_image_csv(
    root_dir='Augmented',
    output_csv='synchronized_swimming_aug.csv'
)

# Imprimir algunas estadísticas del DataFrame resultante
print("Número de imágenes por clase:")
print(data_aug['label'].value_counts())
print()
print("Primeras 5 filas del DataFrame:")
data_aug.head()

Dataset synchronized_swimming_aug.csv guardado con éxito.

Total de imágenes: 6575

Número de imágenes por clase:
label
Knight                             1575
Fishtail                           1475
Bent Knee Vertical                 1300
Double Leg Vertical                1250
Bent Knee Surface Arch Position     975
Name: count, dtype: int64

Primeras 5 filas del DataFrame:


,filepath,label
0,Augmented\Bent Knee Surface Arch Position\IMG_...,Bent Knee Surface Arch Position
1,Augmented\Bent Knee Surface Arch Position\IMG_...,Bent Knee Surface Arch Position
2,Augmented\Bent Knee Surface Arch Position\IMG_...,Bent Knee Surface Arch Position
3,Augmented\Bent Knee Surface Arch Position\IMG_...,Bent Knee Surface Arch Position
4,Augmented\Bent Knee Surface Arch Position\IMG_...,Bent Knee Surface Arch Position
